# 05 · Многодатные слои из Sentinel-2 (GEE): границы по месяцам, СКО NDVI, дата максимума

**Зачем.** Ваш композит — одна «сводка» сезона (max/min/diff NDVI). Соседние поля с одинаковым пиком NDVI, но разной
культурой или датой сева в ней почти неразличимы, а в отдельные месяцы граница между ними видна. Все удачные
классические методы (CEWS, SACFEI, North и др.) сначала ищут границы **на каждой дате**, а потом собирают их по
времени. Этот ноутбук делает то же в GEE и сохраняет результат **на сетке вашего композита**.

**Что получится** — GeoTIFF `<NAME>_multidate.tif` (uint8, та же сетка, что композит):
1. **край-максимум** — максимум по месяцам модуля градиента месячной медианы NDVI (главный слой для конвейера v7);
2. **край-медиана** — медиана того же по месяцам (устойчивые границы);
3. **СКО NDVI** за сезон ×637 (0,4 → 255) — «насколько поле менялось»;
4. **дата максимума NDVI** (день года / 2).

Канал 1 подключается к конвейеру v7: `EXT_TIF = '<путь>/<NAME>_multidate.tif'`, `EXT_BAND = 1`, `EXT_THR = 0.75`.

**Чего ждать (Донбасс, 2026, тест на 301 поле).** Этот слой сам по себе даёт небольшой прирост:
IoU 0,635 → 0,647 (доля IoU≥0,7: 0,55 → 0,57). Границы готовой нейросети FTW дают больше (0,670).
Многодатный слой полезен там, где FTW нет (2026 год) или он плохо видит ваши поля, и как вход для дообучения.

Дополнительно (по желанию) можно выгрузить **сам стек месячных NDVI** (uint8, канал на месяц) — для своих
экспериментов и для небольших участков (для области целиком это десятки ГБ).

**Время:** расчёт идёт на серверах GEE; экспорт области — 20–90 минут. GPU не нужен.

### Подключение Google Диска
Выполните ячейку и разрешите доступ к Диску (всплывёт окно Google). Все результаты пишутся в папку
`Colab_Notebooks/Agro_terra/research/…` — её можно поменять в параметрах ниже.

In [ ]:
import os, sys, json, time, glob, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
LOCAL_TEST = bool(os.environ.get('LOCAL_TEST'))          # True только при проверке вне Colab
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ['LOCAL_DRIVE'])
print('Диск:', DRIVE_ROOT, '| файлов в корне:', len(list(DRIVE_ROOT.iterdir())))

## Шаг 1. Параметры

In [ ]:
EE_PROJECT = 'my-project-20260614-499410'
COMPOSITE_PATH = 'Colab_Notebooks/Agro_terra/TAT/TA_composite_2026_0504_0910_8b.tif'   # сетка результата = сетка композита
AOI_PATH = None                      # необязательно: граница области (иначе — весь охват композита)
YEAR = 2026
MONTHS = [4, 5, 6, 7, 8, 9]          # какие месяцы брать (для юга можно с 3-го по 10-й, для севера — 5–9)
SMOOTH_PX = 1.0                      # сглаживание месячного NDVI перед градиентом, пикс. (0 — без)
EXPORT_MONTHLY_STACK = False         # True — дополнительно выгрузить стек NDVI по месяцам (только для небольших участков!)
OUT_DIR = 'Colab_Notebooks/Agro_terra/research/multidate/out'
GEE_FOLDER = 'GEE_multidate'
NAME = 'TAT_2026'

### Google Earth Engine: подключение и функции сборки многодатных композитов (общие для ноутбуков)
При первом запуске всплывёт окно авторизации Google — разрешите доступ для вашего проекта GEE.

Что умеют функции ниже:
- `s2_collection(aoi, start, end)` — Sentinel-2 L2A **HARMONIZED** (значения уже без смещения +1000, как ждут модели FTW),
  облака/тени маскируются по **Cloud Score+** (`cs_cdf ≥ CS_THR`);
- `window_median(aoi, start, end, bands)` — медиана за окно дат; пропуски (облака) заполняются медианой расширенного окна;
- `monthly_ndvi(aoi, year, months)` — помесячные медианы NDVI (многодатный стек для классических методов);
- `export_blocks(img, aoi, prefix, …)` — экспорт на Google Диск блоками (большой участок режется на квадраты с перекрытием);
- `wait_tasks(tasks)` — ждать окончания экспортов и показывать прогресс.

In [ ]:
import ee, math
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('GEE OK, проект', EE_PROJECT)

S2 = 'COPERNICUS/S2_SR_HARMONIZED'
CSP = 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'
CS_THR = 0.60          # порог Cloud Score+: 0.5 — мягче (больше данных), 0.65 — строже (меньше облаков)


def s2_collection(aoi, start, end, max_cloud=80):
    col = (ee.ImageCollection(S2).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
           .linkCollection(ee.ImageCollection(CSP), ['cs_cdf']))
    return col.map(lambda im: im.updateMask(im.select('cs_cdf').gte(CS_THR)))


def _shift(date, days):
    return ee.Date(date).advance(days, 'day')


def window_median(aoi, start, end, bands, pad_days=30):
    """Медиана за окно [start, end); дыры от облаков заполняются медианой окна, расширенного на ±pad_days."""
    core = s2_collection(aoi, start, end).select(bands).median()
    wide = s2_collection(aoi, _shift(start, -pad_days), _shift(end, pad_days)).select(bands).median()
    return core.unmask(wide)


def monthly_ndvi(aoi, year, months):
    """Многоканальный образ: NDVI-медиана каждого месяца (каналы ndvi_MM), 0–1 → ×10000 int16."""
    out = []
    for m in months:
        start = ee.Date.fromYMD(year, m, 1); end = start.advance(1, 'month')
        nd = s2_collection(aoi, start, end).map(lambda im: im.normalizedDifference(['B8', 'B4']).rename('ndvi')).median()
        out.append(nd.rename(f'ndvi_{m:02d}'))
    return ee.Image.cat(out).multiply(10000).round().toInt16()


def utm_epsg(lon, lat):
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1


def block_grid(bbox, block_deg=0.5, overlap_deg=0.02):
    """Квадраты (по долготе/широте) с перекрытием: [(id, core_bbox, export_bbox)]."""
    x0, y0, x1, y1 = bbox; out = []
    nx, ny = max(1, math.ceil((x1 - x0) / block_deg)), max(1, math.ceil((y1 - y0) / block_deg))
    for i in range(nx):
        for j in range(ny):
            cx0, cy0 = x0 + i * block_deg, y0 + j * block_deg
            core = (cx0, cy0, min(cx0 + block_deg, x1), min(cy0 + block_deg, y1))
            exp = (core[0] - overlap_deg, core[1] - overlap_deg, core[2] + overlap_deg, core[3] + overlap_deg)
            out.append((f'b{i:02d}_{j:02d}', core, exp))
    return out


def export_blocks(img, bbox, prefix, gee_folder, scale=10, block_deg=0.5, dtype='uint16', crs=None):
    """Экспорт изображения на Диск блоками; возвращает список (id, core_bbox, имя_файла, задача).
    Файлы появятся в «Мой диск/<gee_folder>/<prefix>_<id>.tif». crs по умолчанию — UTM центра участка."""
    crs = crs or f'EPSG:{utm_epsg((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)}'
    img = {'uint16': img.toUint16(), 'uint8': img.toUint8(), 'int16': img.toInt16(), 'float': img.toFloat()}[dtype]
    jobs = []
    for bid, core, exp in block_grid(bbox, block_deg):
        name = f'{prefix}_{bid}'
        task = ee.batch.Export.image.toDrive(image=img.clip(ee.Geometry.Rectangle(list(exp))), description=name[:100],
                                             folder=gee_folder, fileNamePrefix=name, region=ee.Geometry.Rectangle(list(exp)),
                                             scale=scale, crs=crs, maxPixels=1e13, fileFormat='GeoTIFF',
                                             fileDimensions=16384)     # один файл на блок (без нарезки на части)
        task.start(); jobs.append((bid, core, name, task))
    print(f'Запущено экспортов: {len(jobs)} (CRS {crs}); следить можно и на https://code.earthengine.google.com/tasks')
    return jobs


def wait_tasks(jobs, poll=30):
    """Ждёт завершения всех задач экспорта. Можно прервать ячейку и вернуться позже — задачи идут на серверах GEE."""
    t0 = time.time()
    while True:
        st = [j[3].status()['state'] for j in jobs]
        done = sum(s == 'COMPLETED' for s in st); bad = [j[2] for j, s in zip(jobs, st) if s in ('FAILED', 'CANCELLED')]
        print(f'\r{time.time() - t0:6.0f} с: готово {done}/{len(jobs)}' + (f', ошибки: {bad}' if bad else ''), end='')
        if done + len(bad) == len(jobs):
            print(); break
        time.sleep(poll)
    for j, s in zip(jobs, st):
        if s == 'FAILED':
            print(j[2], j[3].status().get('error_message'))


def collect_exports(jobs, gee_folder, dst_dir, timeout=900):
    """Переносит экспортированные файлы из «Мой диск/<gee_folder>» в папку проекта (Диск синхронизируется с задержкой)."""
    src_dir = DRIVE_ROOT / gee_folder; dst_dir = Path(dst_dir); dst_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); left = {j[2] for j in jobs}
    while left and time.time() - t0 < timeout:
        for name in list(left):
            for f in src_dir.glob(f'{name}*.tif'):                 # большой блок GEE может разрезать на части …-0000000000-0000000000.tif
                shutil.move(str(f), dst_dir / f.name)
            if list(dst_dir.glob(f'{name}*.tif')):
                left.discard(name)
        if left:
            time.sleep(15)
    print('Файлы в', dst_dir, ':', len(list(dst_dir.glob('*.tif'))), '| не найдено:', sorted(left))

## Шаг 2. Сетка композита
Экспорт идёт с той же проекцией и тем же шагом/началом пикселей (`crsTransform`), что у композита, поэтому
пиксели результата совпадают с пикселями композита один в один.

In [ ]:
import numpy as np, rasterio
from rasterio.warp import transform_bounds
OUT = DRIVE_ROOT / OUT_DIR; OUT.mkdir(parents=True, exist_ok=True)
with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as s:
    C_CRS = s.crs.to_string(); C_T = list(s.transform)[:6]; C_W, C_H = s.width, s.height
    C_BOUNDS = s.bounds; AOI_BBOX = transform_bounds(s.crs, 'EPSG:4326', *s.bounds)
aoi = ee.Geometry.Rectangle(list(AOI_BBOX))
if AOI_PATH:
    import geopandas as gpd
    aoi = ee.Geometry(json.loads(gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326).dissolve().geometry.to_json())['features'][0]['geometry'])
UTM = f'EPSG:{utm_epsg((AOI_BBOX[0] + AOI_BBOX[2]) / 2, (AOI_BBOX[1] + AOI_BBOX[3]) / 2)}'
print('композит', C_W, '×', C_H, 'пикс.,', C_CRS, '| расчёт градиентов в', UTM)

## Шаг 3. Расчёт слоёв в GEE
Для каждого месяца: медиана NDVI безоблачных пикселей (Cloud Score+) → дыры заполняются сезонной медианой →
лёгкое сглаживание → градиент в UTM 10 м → модуль. Затем максимум и медиана по месяцам. Нормировка в 0–255 — по
99,5-му перцентилю по участку (считается в GEE на грубой сетке).

In [ ]:
season = s2_collection(aoi, f'{YEAR}-{MONTHS[0]:02d}-01', ee.Date.fromYMD(YEAR, MONTHS[-1], 1).advance(1, 'month'))
ndvi_col = season.map(lambda im: im.normalizedDifference(['B8', 'B4']).rename('ndvi')
                      .addBands(ee.Image.constant(ee.Date(im.get('system:time_start')).getRelative('day', 'year')).rename('doy').toInt16())
                      .copyProperties(im, ['system:time_start']))
season_med = ndvi_col.select('ndvi').median()
grads, months_nd = [], []
for m in MONTHS:
    st = ee.Date.fromYMD(YEAR, m, 1)
    nd = ndvi_col.filterDate(st, st.advance(1, 'month')).select('ndvi').median().unmask(season_med)
    months_nd.append(nd.rename(f'ndvi_{m:02d}'))
    nd10 = nd.reproject(crs=UTM, scale=10)
    if SMOOTH_PX:
        nd10 = nd10.convolve(ee.Kernel.gaussian(radius=SMOOTH_PX * 2, sigma=SMOOTH_PX, units='pixels'))
    g = nd10.gradient()
    grads.append(g.select('x').hypot(g.select('y')).rename('g'))
G = ee.ImageCollection(grads)
e_max, e_med = G.max().rename('edge_max'), G.median().rename('edge_med')
nd_std = ndvi_col.select('ndvi').reduce(ee.Reducer.stdDev()).rename('ndvi_std')
doy_max = ndvi_col.qualityMosaic('ndvi').select('doy').rename('doy_max')
p = ee.Image.cat([e_max, e_med]).reduceRegion(ee.Reducer.percentile([99.5]), aoi, scale=100, bestEffort=True,
                                              maxPixels=1e9).getInfo()
print('99,5-й перцентиль градиента:', p)
k_max, k_med = 254 / max(p['edge_max'], 1e-9), 254 / max(p['edge_med'], 1e-9)
out_img = ee.Image.cat([e_max.multiply(k_max).add(1), e_med.multiply(k_med).add(1),
                        nd_std.multiply(637).add(1), doy_max.divide(2).add(1)]).clamp(1, 255).toUint8()

## Шаг 4. Экспорт на Диск и сборка в один файл
Экспорт идёт одним файлом на сетке композита (GEE сам режет большие файлы на части; они склеиваются ниже).
Ячейка ждёт окончания; её можно прервать и запустить снова — повторно файл не экспортируется, если уже есть.

In [ ]:
final = OUT / f'{NAME}_multidate.tif'
if not final.exists():
    region = ee.Geometry.Rectangle([C_BOUNDS.left, C_BOUNDS.bottom, C_BOUNDS.right, C_BOUNDS.top], C_CRS, False)
    task = ee.batch.Export.image.toDrive(image=out_img, description=f'{NAME}_multidate', folder=GEE_FOLDER,
                                         fileNamePrefix=f'{NAME}_multidate', region=region, crs=C_CRS, crsTransform=C_T,
                                         maxPixels=1e13, fileDimensions=32768)
    task.start(); jobs = [('all', AOI_BBOX, f'{NAME}_multidate', task)]
    wait_tasks(jobs); collect_exports(jobs, GEE_FOLDER, OUT / 'parts')
    parts_ = sorted((OUT / 'parts').glob(f'{NAME}_multidate*.tif'))
    prof = dict(driver='GTiff', width=C_W, height=C_H, count=4, dtype='uint8', crs=C_CRS,
                transform=rasterio.Affine(*C_T), compress='deflate', tiled=True, BIGTIFF='IF_SAFER', nodata=0)
    with rasterio.open(final, 'w', **prof) as dst:
        for pth in parts_:
            with rasterio.open(pth) as srcp:
                win = rasterio.windows.from_bounds(*srcp.bounds, dst.transform).round_offsets().round_lengths()
                dst.write(srcp.read(window=rasterio.windows.Window(0, 0, win.width, win.height)), window=win)
    print('Сохранено:', final, '| частей:', len(parts_))
else:
    print('Уже есть:', final)

## Шаг 5 (по желанию). Стек месячных NDVI
Канал на месяц, uint8 (NDVI −0,2…1 → 1…255). Только для небольших участков: для области целиком файл огромный.

In [ ]:
if EXPORT_MONTHLY_STACK:
    stack = ee.Image.cat(months_nd).add(0.2).multiply(254 / 1.2).add(1).clamp(1, 255).toUint8()
    region = ee.Geometry.Rectangle([C_BOUNDS.left, C_BOUNDS.bottom, C_BOUNDS.right, C_BOUNDS.top], C_CRS, False)
    t = ee.batch.Export.image.toDrive(image=stack, description=f'{NAME}_ndvi_monthly', folder=GEE_FOLDER,
                                      fileNamePrefix=f'{NAME}_ndvi_monthly', region=region, crs=C_CRS, crsTransform=C_T,
                                      maxPixels=1e13, fileDimensions=32768)
    t.start(); jobs = [('m', AOI_BBOX, f'{NAME}_ndvi_monthly', t)]
    wait_tasks(jobs); collect_exports(jobs, GEE_FOLDER, OUT)

## Шаг 6. Быстрый просмотр
Фрагмент 10×10 км в центре: композит и край-максимум рядом.

In [ ]:
import matplotlib.pyplot as plt
with rasterio.open(final) as s, rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
    cy, cx = s.height // 2, s.width // 2; w = rasterio.windows.Window(cx - 700, cy - 700, 1400, 1400)
    e = s.read(1, window=w); comp = c.read([1, 2, 3], window=w).transpose(1, 2, 0)
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
ax[0].imshow(comp); ax[0].set_title('композит'); ax[1].imshow(e, cmap='magma', vmin=0, vmax=255); ax[1].set_title('край-максимум по месяцам')
for a in ax: a.axis('off')
plt.savefig(OUT / f'{NAME}_multidate_view.png', dpi=110, bbox_inches='tight'); plt.show()

## Что дальше
В ноутбуке конвейера v7 в пресете (или в `BASE`) поставьте:
`EXT_TIF='…/research/multidate/out/<NAME>_multidate.tif', EXT_BAND=1, EXT_THR=0.75`.
Порог удобно подобрать автоматически ноутбуком 06 (укажите этот файл в `EXT_BOUNDARY_TIF`).
Несколько источников границ объединяются списком, например FTW + многодатные:
`EXT_TIF=[dict(TIF='…/TAT_ftw_prob.tif', BAND=1, THR=0.3), dict(TIF='…/TAT_2026_multidate.tif', BAND=1, THR=0.75)]`
(на Донбассе: только FTW — IoU 0,670; FTW + многодатные — 0,671; FTW + суперпиксели Higra — 0,676).